# Day 2: Dense Embeddings & Qdrant Metadata Filtering (Visual X-Ray)
**Config Source:** `.env` + `configs/app_config.yaml` (`engine/settings.py`) — Zero hardcoded URLs in notebook cells.


### Cell 1 — Centralized Settings & Modular Engine Bootstrap


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from engine.settings import SETTINGS
from engine.router import ModelRouter
from engine.qdrant_store import QdrantVectorStore
from engine.use_case_loader import UseCaseLoader
from engine.visualizer import StoryVisualizer
from engine.debugger import TraceDebugger

# Import PromptRegistry safely regardless of module filename (prompts.py or prompt_registry.py)
try:
    from engine.prompts import PromptRegistry
except ImportError:
    from engine.prompt_registry import PromptRegistry

# All URLs, paths, domain names, and cloud toggles come from SETTINGS (.env + configs/app_config.yaml)
ACTIVE_DOMAIN = SETTINGS.active_domain  # Override here or via BOOTCAMP_DOMAIN in .env

loader = UseCaseLoader(domain=ACTIVE_DOMAIN)
llm = ModelRouter(local_model_path=SETTINGS.local_model_path, enable_vertex=SETTINGS.enable_vertex_ai)
qdrant = QdrantVectorStore(collection_name=loader.collection_name, path=SETTINGS.qdrant_db_path)
prompts = PromptRegistry()
viz = StoryVisualizer()
dbg = TraceDebugger(debug_mode=False, enable_phoenix=False)

RUNBOOKS = loader.load_runbooks()
INCIDENTS = loader.load_incidents()
qdrant.upsert_documents(RUNBOOKS)

print(f"[Ready] Domain: {loader.domain} | LLM: {llm.provider} | VectorDB: {qdrant.mode} ({loader.collection_name})")


### Cell 2 — Production Incident Selector (0..9)


In [ ]:
# Select any of the 10 production incidents in the active domain (0..9)
INCIDENT_INDEX = 0
ACTIVE_INCIDENT = INCIDENTS[INCIDENT_INDEX]

print(f"Selected Incident: [{ACTIVE_INCIDENT['incident_id']}] {ACTIVE_INCIDENT['title']} (Severity: {ACTIVE_INCIDENT['severity']})")
print("Raw Log:", ACTIVE_INCIDENT["raw_log"])


### Cell 3 — Engineering Concept Brief
**Concept: Semantic Similarity + Metadata Payload Filters**
Compares global Qdrant cosine search against category-filtered search using externalized domain runbooks.


### Cell 4 — Traced Execution & X-Ray Visual Storyboard


In [ ]:
dbg.start_run("day02_qdrant_search")
query_text = f"{ACTIVE_INCIDENT['title']} - {ACTIVE_INCIDENT['raw_log']}"

with dbg.start_span(f"Day2_QdrantSearch_{ACTIVE_INCIDENT['incident_id']}", span_kind="CHAIN") as root_span:
    t0 = time.perf_counter()
    unfiltered_hits = qdrant.search(query=query_text, top_k=3)
    t_unf = round((time.perf_counter() - t0) * 1000, 2)
    dbg.record_step("qdrant_unfiltered_search", {"query": query_text}, unfiltered_hits, t_unf, span_kind="RETRIEVER")

    t1 = time.perf_counter()
    cat_filter = {"category": ACTIVE_INCIDENT["category"]}
    filtered_hits = qdrant.search(query=query_text, top_k=2, metadata_filter=cat_filter)
    t_flt = round((time.perf_counter() - t1) * 1000, 2)
    dbg.record_step("qdrant_filtered_search", {"query": query_text, "filter": cat_filter}, filtered_hits, t_flt, span_kind="RETRIEVER")

viz.render_story(
    title=f"Day 2 Qdrant Vector Radar — {ACTIVE_INCIDENT['incident_id']}",
    stages=[
        {"name": "Unfiltered Qdrant Search (Top-3)", "badge": "GLOBAL SEARCH", "latency_ms": t_unf, "summary": "All domain runbooks ranked by cosine similarity", "scores": [{"label": h["metadata"]["runbook_id"], "score": h["score"]} for h in unfiltered_hits], "payload": unfiltered_hits[0]},
        {"name": f"Filtered Qdrant Search (filter={cat_filter})", "badge": "FILTERED SEARCH", "latency_ms": t_flt, "summary": "Payload filter isolates exact category runbooks", "scores": [{"label": h["metadata"]["runbook_id"], "score": h["score"]} for h in filtered_hits], "payload": filtered_hits}
    ],
    phoenix_url=dbg.phoenix_url
)


### Cell 5 — Guarded Cloud Parity Check (`SETTINGS.enable_vertex_ai`)


In [ ]:
if SETTINGS.enable_vertex_ai:
    print("Vertex AI Vector Search parity mode active.")
else:
    print("ENABLE_VERTEX_AI=false -> Local Qdrant metadata filtering verified.")


### Cell 6 — Automated Validation Gate


In [ ]:
assert any(h["metadata"]["runbook_id"] == ACTIVE_INCIDENT["expected_runbook"] for h in filtered_hits)
print("[PASS] Day 2 Qdrant Metadata Filtering Gate passed!")
